In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer


data = pd.read_csv("../Data/Raw/train.csv")
test_data = pd.read_csv("../Data/Raw/test.csv")

numerical_data = [
    "sleep_duration",
    "heart_rate",
    "bmi",
    "step_count",
    "exercise_duration",
    "water_intake",
    "calorie_expenditure"
]


base_categorical_data = [
    "diet_type",
    "stress_level",
    "sleep_quality",
    "physical_activity_level",
    "smoking_alcohol",
    "gender"
]

num_imputer = SimpleImputer(strategy="median")
data[numerical_data] = num_imputer.fit_transform(data[numerical_data])


cat_imputer = SimpleImputer(strategy="most_frequent")
data[base_categorical_data] = cat_imputer.fit_transform(data[base_categorical_data])

test_data[numerical_data] = num_imputer.transform(test_data[numerical_data])
test_data[base_categorical_data] = cat_imputer.transform(test_data[base_categorical_data])


def bmi_category(bmi):
    if bmi < 16:
        return "Severe_Thinness"
    elif bmi < 18.5:
        return "Underweight"
    elif bmi < 22:
        return "Normal_Low"
    elif bmi < 25:
        return "Normal_High"
    elif bmi < 27.5:
        return "Overweight_Low"
    elif bmi < 30:
        return "Overweight_High"
    elif bmi < 35:
        return "Obese"
    else:
        return "Severe_Obese"


def heart_rate_category(hr):
    if hr < 60:
        return "Low"
    elif hr <= 120:
        return "Normal"
    else:
        return "High"


def activity_category(steps):
    if steps < 5000:
        return "Low"
    elif steps < 10000:
        return "Moderate"
    else:
        return "High"


def exercise_category(exercise):
    if exercise == 0:
        return "None"
    elif exercise < 30:
        return "Short"
    elif exercise < 150:
        return "Normal"
    else:
        return "High"


def water_category(water):
    if water < 1.5:
        return "Low"
    elif water < 3:
        return "Normal"
    else:
        return "High"

test_data["bmi_category"] = test_data["bmi"].apply(bmi_category)

test_data["heart_rate_category"] = (
    test_data["heart_rate"]
    .apply(heart_rate_category)
)

test_data["activity_category"] = (
    test_data["step_count"]
    .apply(activity_category)
)

test_data["exercise_category"] = (
    test_data["exercise_duration"]
    .apply(exercise_category)
)

test_data["water_category"] = (
    test_data["water_intake"]
    .apply(water_category)
)


# Feature engineering
data["bmi_category"] = data["bmi"].apply(bmi_category)
data["heart_rate_category"] = data["heart_rate"].apply(heart_rate_category)
data["activity_category"] = data["step_count"].apply(activity_category)
data["exercise_category"] = data["exercise_duration"].apply(exercise_category)
data["water_category"] = data["water_intake"].apply(water_category)

categorical_data = [
    "diet_type",
    "stress_level",
    "sleep_quality",
    "physical_activity_level",
    "smoking_alcohol",
    "gender",
    "bmi_category",
    "heart_rate_category",
    "activity_category",
    "exercise_category",
    "water_category"
]


y = data["health_condition"]
X = data[numerical_data + categorical_data]
X_submission = test_data[numerical_data + categorical_data]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_data
        )
    ],
    remainder="passthrough"
)


model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
    ))
])

param_grid = {
    "classifier__n_estimators": [100, 200, 300, 500, 800],
    
    "classifier__max_depth": [
        None,
        5,
        10,
        20,
        30
    ],
    
    "classifier__min_samples_split": [
        2,
        5,
        10
    ],
    
    "classifier__min_samples_leaf": [
        1,
        2,
        5,
        10
    ],
    
    "classifier__max_features": [
        "sqrt",
        "log2",
        None
    ],
    
    "classifier__bootstrap": [
        True,
        False
    ]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)


grid_search.fit(X_train, y_train)
model.fit(X_train, y_train)


scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

print("Best Parameters:")
print(grid_search.best_params_)

print("Best CV:")
print(grid_search.best_score_)


best_model = grid_search.best_estimator_

predictions = best_model.predict(X_submission)

submission = pd.DataFrame({
    "id": test_data["id"],
    "health_condition": predictions
})

submission.to_csv(
    r"C:\Users\danie\Downloads\submission.csv",
    index=False
)


print("CV Accuracy:", scores)
print("Mean:", scores.mean())
print("Std:", scores.std())

Fitting 5 folds for each of 1800 candidates, totalling 9000 fits
